# Hardened substrate sanity — can d=96 learn degree=16 PCFG?

**One question only:** on a `LEVELS=6, NT=4, T=4, degree=16` PCFG (Allen-Zhu-style
pigeonhole-DP hardness — 16 rules per NT with only 4 possible first-children
forces most rules to share their first child, so first-token-of-production
carries almost no info about which rule fired), does a `d=96` TinyRotaryGPT
land in the capacity-constrained regime for monolingual next-token prediction?

Target band: `acc ∈ (0.5, 0.85)`.
- `acc < 0.5` → grammar too hard at this scale; back off to degree=12 or 8.
- `acc > 0.85` → still too easy; push degree=24 or NT_SIZE=3.
- in-band → proceed to build the full `ρ × data × capacity` factorial on this grammar.

**Nothing else runs here.** No conflict map, no multilingual mix, no factorial — those
come after sanity passes. Online fresh tree sampling throughout (no reused tree_bank,
no repetition).

Set `GPU_ID` in cell 1.

In [1]:
import os, time, math, random, statistics as _st
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext

GPU_ID = 1
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  {torch.cuda.get_device_name(GPU_ID)}")

def autocast_ctx():
    return (torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16)
            if DEVICE.type == "cuda" else nullcontext())

torch.manual_seed(0)
random.seed(0)

using device: cuda:1
  NVIDIA H200 NVL


## 1. Config

In [2]:
# Allen-Zhu pigeonhole-DP hardness lever: degree pushed past the terminal alphabet.
# With T=NT=4 and degree=16, pigeonhole forces ~99% first-child prefix overlap
# per NT (verified in prior calibration pass), so a parser can't greedy-shortcut
# off the first child; it has to actually track hierarchy.
# Keeps LEVELS=6 / SEQ_LEN=512 — no 4x compute hit of cfg3f-exact's SEQ_LEN=1024.

LEVELS       = 6
NT_SIZE      = 4
TERM_SIZE    = 4
MIN_DEGREE   = 4
MAX_DEGREE   = 4
RULE_LEN_CHOICES = [2, 3]

# Languages set up for later; sanity uses L_A only.
LANGS = {
    "L_A": set(),                       # anchor: identity linearization
    "L_B": {4, 5},                      # partial reversal (unused in sanity)
    "L_C": set(range(1, LEVELS + 1)),   # full reversal     (unused in sanity)
}
LANG_NAMES = list(LANGS.keys())
ANCHOR = "L_A"

BOS_OFFSET = TERM_SIZE
BOS = {name: BOS_OFFSET + i for i, name in enumerate(LANG_NAMES)}
VOCAB_SIZE = TERM_SIZE + len(LANG_NAMES)

N_HEAD      = 4
N_LAYER     = 8
SEQ_LEN     = 512
BATCH_SIZE  = 32
LR          = 3e-3
STEP_BUDGET = 8000

D_SANITY    = 96     # the one capacity we're checking
SEED_SANITY = 0

for d in [D_SANITY]:
    assert d % N_HEAD == 0 and (d // N_HEAD) % 2 == 0, (
        f"d={d}: head_dim must be even for rotary")

print(f"grammar: LEVELS={LEVELS}  NT={NT_SIZE}  T={TERM_SIZE}  "
      f"degree={MIN_DEGREE}-{MAX_DEGREE}  rule_lens={RULE_LEN_CHOICES}")
print(f"vocab={VOCAB_SIZE}  seq_len={SEQ_LEN}  batch={BATCH_SIZE}  "
      f"steps={STEP_BUDGET}  lr={LR}")
print(f"model: d={D_SANITY} x {N_LAYER} layers x {N_HEAD} heads  "
      f"(seed={SEED_SANITY})")

grammar: LEVELS=6  NT=4  T=4  degree=4-4  rule_lens=[2, 3]
vocab=7  seq_len=512  batch=32  steps=8000  lr=0.003
model: d=96 x 8 layers x 4 heads  (seed=0)


## 2. Grammar, sampler, linearizer

In [3]:
def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = (terminals if child_level == LEVELS + 1
                         else symbols_by_level[child_level])
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set()
            tries = 0
            while len(rules[sym]) < degree and tries < 5000:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs)
                rules[sym].append(rhs)
            if len(rules[sym]) < degree:
                raise RuntimeError(
                    f"could only build {len(rules[sym])}/{degree} unique rules "
                    f"for symbol {sym}; raise rule_lens or lower degree")
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens

def _nleaves(n):
    return 1 if isinstance(n, int) else sum(_nleaves(c) for c in n[1])

substrate_seed = 0
rules, symbols_by_level = build_grammar(substrate_seed)

rule_counts = [len(rules[s]) for lvl in symbols_by_level
               for s in symbols_by_level[lvl]]
print(f"built {sum(rule_counts)} rules across {len(rule_counts)} NTs  "
      f"(min={min(rule_counts)} max={max(rule_counts)})")

# Tree-length distribution on this grammar (sanity: all trees fit SEQ_LEN).
_rng = random.Random(42)
sample_lens = sorted(_nleaves(sample_tree(rules, _rng)) for _ in range(2000))
pct = {p: sample_lens[int(p * len(sample_lens)) - 1] for p in [0.5, 0.95, 0.99]}
fit_frac = sum(l <= SEQ_LEN - 2 for l in sample_lens) / len(sample_lens)
print(f"tree lengths  median={pct[0.5]}  95th={pct[0.95]}  99th={pct[0.99]}  "
      f"max={sample_lens[-1]}")
print(f"fit SEQ_LEN={SEQ_LEN} (minus BOS+1): {fit_frac:.1%}")
assert fit_frac >= 0.98, f"too many trees overflow SEQ_LEN={SEQ_LEN}; raise SEQ_LEN"
print("  OK: tree lengths fit")

built 84 rules across 21 NTs  (min=4 max=4)
tree lengths  median=269  95th=371  99th=401  max=434
fit SEQ_LEN=512 (minus BOS+1): 100.0%
  OK: tree lengths fit


## 3. Decodability gate

Exp6-style: for every sampled tree, record `(terminal_string, derivation_signature)`.
Two trees with the same terminal string but different signatures are a genuine
ambiguity witness. 50 k samples; same empirical bar as Allen-Zhu's own
birthday-paradox check. Can only catch pervasive ambiguity, not rare collisions —
formal unambiguity for an arbitrary CFG is undecidable.

In [4]:
def sample_tree_labeled(rules, rng, symbol=(1, 0), level=1):
    rule_idx = rng.randrange(len(rules[symbol]))
    rhs = rules[symbol][rule_idx]
    sig = [(symbol, rule_idx)]
    toks = []
    for child_sym in rhs:
        if level == LEVELS:
            toks.append(child_sym)
        else:
            ct, cs = sample_tree_labeled(rules, rng, symbol=child_sym, level=level + 1)
            toks.extend(ct)
            sig.extend(cs)
    return toks, sig

print(f"[decodability] sampling 50000 trees from degree={MIN_DEGREE} grammar...")
t0 = time.time()
seen = {}
collisions = 0
_rng = random.Random(7)
for i in range(50000):
    toks, sig = sample_tree_labeled(rules, _rng)
    key = tuple(toks)
    sig_t = tuple(sig)
    if key in seen:
        if seen[key] != sig_t:
            collisions += 1
    else:
        seen[key] = sig_t
print(f"  {collisions} ambiguous collisions / 50000 samples  "
      f"({len(seen)} unique strings, {time.time()-t0:.1f}s)")
assert collisions == 0, (
    f"grammar has pervasive ambiguity at degree={MIN_DEGREE}; "
    f"back off degree or raise NT_SIZE")
print(f"  OK: decodability gate passes at degree={MIN_DEGREE}")

[decodability] sampling 50000 trees from degree=4 grammar...
  0 ambiguous collisions / 50000 samples  (50000 unique strings, 5.0s)
  OK: decodability gate passes at degree=4


## 4. Online fresh-batch sampler (no tree_bank, no repetition)

In [5]:
def online_batch(rules, lang, batch_size, seq_len, rng):
    reverse_levels = LANGS[lang]
    bos = BOS[lang]
    target_len = batch_size * (seq_len + 1)
    stream = []
    while len(stream) < target_len:
        tree = sample_tree(rules, rng)
        stream.append(bos)
        stream.extend(linearize(tree, reverse_levels))
    stream = stream[:target_len]
    return torch.tensor(stream, dtype=torch.long).view(batch_size, seq_len + 1)

# smoke
_rng = random.Random(99)
_b = online_batch(rules, ANCHOR, 4, 64, _rng)
print(f"online_batch smoke  shape={tuple(_b.shape)}  "
      f"row0[:12]={_b[0,:12].tolist()}  BOS[{ANCHOR}]={BOS[ANCHOR]}")

online_batch smoke  shape=(4, 65)  row0[:12]=[4, 0, 3, 1, 0, 3, 1, 1, 3, 2, 3, 1]  BOS[L_A]=4


## 5. TinyRotaryGPT (standard 8L / 4H / rotary)

In [6]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return ((q * cos) + (rotate_half(q) * sin),
            (k * cos) + (rotate_half(k) * sin))

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd,
                 n_layer=N_LAYER, n_head=N_HEAD, max_seq_len=SEQ_LEN + 1):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def count_params(d):
    m = TinyRotaryGPT(VOCAB_SIZE, n_embd=d)
    return sum(p.numel() for p in m.parameters())

# forward smoke
_m = TinyRotaryGPT(VOCAB_SIZE, n_embd=8, n_layer=2, n_head=2, max_seq_len=64).to(DEVICE)
_idx = torch.randint(0, VOCAB_SIZE, (4, 32), device=DEVICE)
print("forward OK:", _m(_idx, _idx)[0].shape, "params(d=96):", f"{count_params(96):,}")

forward OK: torch.Size([4, 32, 7]) params(d=96): 896,256


## 6. Training + eval (online fresh batches throughout)

In [7]:
def train_online(d_model, lang, step_budget, seed, log_every=500):
    torch.manual_seed(seed)
    rng = random.Random(seed + 1000)
    model = TinyRotaryGPT(VOCAB_SIZE, n_embd=d_model).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    losses = []
    t0 = time.time()
    for step in range(step_budget):
        batch = online_batch(rules, lang, BATCH_SIZE, SEQ_LEN, rng).to(DEVICE)
        x, y = batch[:, :-1], batch[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if (step + 1) % log_every == 0:
            tail = _st.mean(losses[-100:])
            print(f"  step {step+1:5d}/{step_budget}  "
                  f"tail-100 loss={tail:.4f}  ({time.time()-t0:.0f}s)")
    return model, losses

@torch.no_grad()
def eval_accuracy_online(model, lang, n_batches=50, seed=12345):
    model.eval()
    rng = random.Random(seed)
    correct, total = 0, 0
    for _ in range(n_batches):
        batch = online_batch(rules, lang, BATCH_SIZE, SEQ_LEN, rng).to(DEVICE)
        x, y = batch[:, :-1], batch[:, 1:]
        with autocast_ctx():
            logits, _ = model(x, y)
        pred = logits.argmax(-1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

## 7. Sanity run — d=96 monolingual L_A on degree=16

Verdict band: `acc ∈ (0.5, 0.85)` is capacity-constrained. Below → grammar too hard
at d=96 (back off degree=12 or 8). Above → still too easy (push degree=24 or
drop NT_SIZE=3). In-band → proceed to build the ρ × data × capacity factorial on
this grammar.

In [8]:
print(f"[sanity] d={D_SANITY}  lang={ANCHOR}  degree={MIN_DEGREE}  "
      f"steps={STEP_BUDGET}  seed={SEED_SANITY}")
print(f"  params: {count_params(D_SANITY):,}  chance acc: {1/VOCAB_SIZE:.3f}  "
      f"(vocab={VOCAB_SIZE})")
print()

t0 = time.time()
model, losses = train_online(D_SANITY, ANCHOR, STEP_BUDGET, seed=SEED_SANITY)
train_min = (time.time() - t0) / 60.0

t0 = time.time()
acc = eval_accuracy_online(model, ANCHOR)
eval_s = time.time() - t0

final_loss = _st.mean(losses[-100:])
chance = 1.0 / VOCAB_SIZE

print()
print("=" * 68)
print(f"d={D_SANITY}  {ANCHOR}  degree={MIN_DEGREE}  "
      f"final_loss={final_loss:.4f}  acc={acc:.3f}  (chance={chance:.3f})")
print(f"  train: {train_min:.1f} min   eval: {eval_s:.1f} s")
print("=" * 68)

if acc < 0.5:
    print(f"VERDICT: TOO HARD at d={D_SANITY}. "
          f"Back off: try MIN_DEGREE=MAX_DEGREE=12 (or 8) and rerun this notebook.")
elif acc > 0.85:
    print(f"VERDICT: TOO EASY at d={D_SANITY}. "
          f"Push harder: try MIN_DEGREE=MAX_DEGREE=24, or NT_SIZE=3, and rerun.")
else:
    print(f"VERDICT: IN-BAND (capacity-constrained regime). "
          f"Proceed to build the ρ × data × capacity factorial on this grammar.")

[sanity] d=96  lang=L_A  degree=4  steps=8000  seed=0
  params: 896,256  chance acc: 0.143  (vocab=7)

  step   500/8000  tail-100 loss=1.0470  (9s)
  step  1000/8000  tail-100 loss=1.0401  (19s)
  step  1500/8000  tail-100 loss=1.0077  (27s)
  step  2000/8000  tail-100 loss=0.9910  (37s)
  step  2500/8000  tail-100 loss=0.9824  (46s)
  step  3000/8000  tail-100 loss=0.9775  (56s)
  step  3500/8000  tail-100 loss=0.9953  (66s)
  step  4000/8000  tail-100 loss=0.9702  (75s)
  step  4500/8000  tail-100 loss=0.9663  (84s)
  step  5000/8000  tail-100 loss=0.9635  (94s)
  step  5500/8000  tail-100 loss=0.9613  (104s)
  step  6000/8000  tail-100 loss=0.9596  (113s)
  step  6500/8000  tail-100 loss=0.9592  (123s)
  step  7000/8000  tail-100 loss=0.9572  (133s)
  step  7500/8000  tail-100 loss=0.9569  (142s)
  step  8000/8000  tail-100 loss=0.9551  (152s)

d=96  L_A  degree=4  final_loss=0.9551  acc=0.554  (chance=0.143)
  train: 2.5 min   eval: 0.5 s
VERDICT: IN-BAND (capacity-constrained reg